In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from build_spectra.linear_combination import generate_mixture_spectra

rng = np.random.default_rng(seed=None)

# Set a nice default style
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 12

# Preferably use a GPU instead of CPU
print(f"PyTorch version: {torch.__version__}")
# --- Auto-detect GPU (CUDA or Apple MPS) — falls back to CPU ---
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

In [ ]:
df = pd.read_excel('../../../data/spectral_library_with_scattering.xlsx')
df_clean = pd.read_excel('../../../data/spectral_library_clean.xlsx')

df.ffill(axis=0, inplace=True)
df.bfill(axis=0, inplace=True)  # handles leading NaNs

# if there is a NaN, interpolate data linearly

wavelength = df['Wavelength']

In [ ]:
class Sparsemax(nn.Module):
    def forward(self, z):
        dim = 1
        z_sorted, _ = torch.sort(z, dim=dim, descending=True)
        
        css = torch.cumsum(z_sorted, dim=dim)

        z_index = torch.arange(1, z.shape[dim] + 1, device=z.device).float()
        bound = 1 + z_index * z_sorted > css
        k = bound.sum(dim=dim, keepdim=True).float()
        
        tau = (torch.gather(css, dim, (k - 1).long()) - 1) / k
        
        return torch.clamp(z - tau, min=0.0)

def training_loop(nr_epochs, epochs_no_improvement, X_train_, y_train_, X_val_, y_val_, wavelength):  
    N_species = y_train_.shape[1] 

    print(X_train_.shape, X_val_.shape)
    
    X_train_ = X_train_.float()
    X_val_ = X_val_.float()

    model = nn.Sequential(
        nn.Linear(len(wavelength), 128),
        nn.ReLU(),
        nn.Linear(128, N_species),
        Sparsemax()
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5, min_lr=1e-6)

    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    train_maes = []
    val_maes = []
    epochs_list = []

    X_train_ = X_train_.reshape(-1, len(wavelength)).float()
    X_val_ = X_val_.reshape(-1, len(wavelength)).float()
    
    y_train_ = y_train_.reshape(-1, N_species).float()
    y_val_ = y_val_.reshape(-1, N_species).float()

    best_val_loss = float('inf')
    epochs_without_improvement = 0

    for epoch in range(nr_epochs):
        model.train()
        optimizer.zero_grad()

        pred = model(X_train_)
        
        loss = criterion(pred, y_train_)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            # dim=0 averages across the 4000 samples, leaving 9 species MAEs
            train_mae_per_species = torch.mean(torch.abs(pred - y_train_), dim=0).tolist()

        model.eval()
        with torch.no_grad():
            pred_val = model(X_val_)
            loss_val = criterion(pred_val, y_val_)

            val_mae_per_species = torch.mean(torch.abs(pred_val - y_val_), dim=0).tolist()

        train_losses.append(loss.item())
        val_losses.append(loss_val.item()) 
        
        train_maes.append(train_mae_per_species)
        val_maes.append(val_mae_per_species)

        if loss_val.item() < best_val_loss:
            best_val_loss = loss_val.item()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement > epochs_no_improvement:
            print(f"Early stopping triggered at epoch {epoch}")
            break
            
        scheduler.step(loss_val.item())
        
    return train_losses, val_losses, train_maes, val_maes, epochs_list, model

In [ ]:
mixtures_df, weights_df = generate_mixture_spectra(df, combination_sizes=(3, 4, 5, 6, 7, 8, 9), n_mixtures_per_combination=100)

mixture_columns = weights_df['mixture_name'].tolist()
X_numpy = mixtures_df[mixture_columns].T.values 

y_df = weights_df.drop(columns=['mixture_name'])
y_numpy = y_df.values

X_train, X_val, y_train, y_val = train_test_split(X_numpy, y_numpy, test_size=0.2, random_state=42)

X_train_ = torch.tensor(X_train, dtype=torch.float32)
X_val_ = torch.tensor(X_val, dtype=torch.float32)
y_train_ = torch.tensor(y_train, dtype=torch.float32)
y_val_ = torch.tensor(y_val, dtype=torch.float32)

noise_level = 0.1

train_noise = noise_level * torch.randn_like(X_train_)
X_train_ += train_noise

val_noise = noise_level * torch.randn_like(X_val_)
X_val_ += val_noise

# Normalize X
mean = X_train_.mean(dim=0, keepdim=True)
std = X_train_.std(dim=0, keepdim=True) + 1e-8
X_train_ = (X_train_ - mean) / std
X_val_ = (X_val_ - mean) / std

In [ ]:
train_losses, val_losses, train_maes, val_maes, epochs_list, trained_model = training_loop(10000, 8, X_train_, y_train_, X_val_, y_val_, wavelength)

In [ ]:
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

trained_model.eval()
with torch.no_grad():
    pred_val = trained_model(X_val)
    
    # Calculate absolute error for every prediction
    abs_errors = torch.abs(pred_val - y_val)

# Calculate 
per_mixture_mae = torch.mean(abs_errors, dim=1).cpu().numpy()
per_mixture_std = torch.std(abs_errors, dim=1).cpu().numpy()
overall_mean_mae = np.mean(per_mixture_mae)

per_species_mae = torch.mean(abs_errors, dim=0).cpu().numpy()
per_species_std = torch.std(abs_errors, dim=0).cpu().numpy()

species_names = [
    "Diatom_Ptricornutum", "Diatom_Csimplex", 
    "Chlamydomonas_Cprisculi", "Chlamydomonas_Creindhardtii", 
    "Dinoflagellate_Symbiodiniumsp", "Dinoflagellate_Smicroadriaticum",
    "Dinoflagellate_Dtrenchii", "Dinoflagellate_Cgoreaui",
    "Cyanobacteria_Synechosystis"
]

In [ ]:
print(pred_val.shape)

In [ ]:
train_maes_np = np.array(train_maes)
val_maes_np = np.array(val_maes)

print(val_maes_np.shape)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i in range(9):
    axes[i].plot(train_maes_np[:, i], label='Train MAE')
    axes[i].plot(val_maes_np[:, i], label='Val MAE')
    
    axes[i].set_title(f'{species_names[i]}')
    axes[i].set_xlabel('Epochs')
    axes[i].set_ylabel('MAE')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

ax1 = axes[0]
mixture_indices = np.arange(len(per_mixture_mae))

ax1.plot(mixture_indices, per_mixture_mae, marker='.', linestyle='-', lw=0.5, markersize=6, color='tab:blue')

ax1.fill_between(mixture_indices, 
                 per_mixture_mae - per_mixture_std, 
                 per_mixture_mae + per_mixture_std, 
                 alpha=0.3, color='tab:blue', label='±1 std')

ax1.axhline(overall_mean_mae, color='red', linestyle='--', lw=1.5, label=f'Mean MAE = {overall_mean_mae:.3f}')

ax1.set_title("Per-mixture mean absolute error", fontsize=14)
ax1.set_xlabel("Mixture index", fontsize=11)
ax1.set_ylabel("MAE", fontsize=11)
ax1.legend(loc='upper left', fontsize=11)

ax2 = axes[1]
x_pos = np.arange(len(species_names))

ax2.bar(x_pos, per_species_mae, yerr=per_species_std, capsize=5, color='tab:blue')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(species_names, rotation=45, ha="right")

ax2.set_title("Per-species mean absolute error", fontsize=14)
ax2.set_ylabel("MAE", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
print(mixtures_df.shape)
print(mixtures_df)

In [ ]:
# Adapt some variables for code below
weights = weights_df.copy()
weights = weights_df.drop(['mixture_name'], axis=1).to_numpy()
print(weights.shape)

X_all_tensor = torch.tensor(X_numpy, dtype=torch.float32)
X_all_tensor = (X_all_tensor - mean) / std

trained_model.eval()
with torch.no_grad():
    pred_all = trained_model(X_all_tensor)

pred_all_np = pred_all.cpu().numpy()

estimated_concentrations = pd.DataFrame(
    pred_all_np, 
    columns=species_names, 
).to_numpy()

print(estimated_concentrations.shape)

In [ ]:
# ground_truth shape: (4600, 9), estimated_concentrations shape: (4600, 9)
# Identify dominant species per mixture (highest true weight)
dominant_species = np.argmax(weights, axis=1)  # (4600,)

n_species = weights.shape[1]
heatmap = np.zeros((n_species, n_species))
species_names = df_clean.columns[1:].tolist()

for i in range(n_species):
    # Select all mixtures where species i is dominant
    mask = dominant_species == i
    if mask.sum() > 0:
        # Average predicted concentration of each species j in those mixtures
        heatmap[i, :] = estimated_concentrations[mask].mean(axis=0)

# Optional: normalize each row so values sum to 1
heatmap_norm = heatmap / heatmap.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    heatmap_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=species_names,
    yticklabels=species_names,
    ax=ax,
    vmin=0, vmax=1
)
ax.set_xlabel("Predicted species (mean estimated concentration)")
ax.set_ylabel("Dominant true species")
ax.set_title("MLP: Mean predicted concentration\ngrouped by dominant true species")
plt.tight_layout()
plt.show()
# this does not show how the model performs. This plot picks dominant species. Dominant species can have weights of .31, so the spectra is still dominated by the other species! 
# That's why this does not show high diagonal values. To fix this, use a mask with a threshold for the weights -- only dominant species with weights > .6-.8 will show. 


In [ ]:
threshold = 0.7  # adjust as needed

# Print sample counts per species at each threshold
print("Sample counts per species at different thresholds:")
for thresh in [0.4, 0.5, 0.6, 0.7]:
    mask = weights.max(axis=1) > thresh
    counts = np.bincount(np.argmax(weights[mask], axis=1), minlength=9)
    print(f"  thresh={thresh}: {counts} (total={mask.sum()})")
print(weights[:5])
# Apply threshold: only keep mixtures where dominant species > threshold
mask_dominant = weights.max(axis=1) > threshold
gt_filtered   = weights[mask_dominant]
ec_filtered   = estimated_concentrations[mask_dominant]

dominant_species = np.argmax(gt_filtered, axis=1)

n_species = weights.shape[1]
heatmap = np.zeros((n_species, n_species))

for i in range(n_species):
    mask = dominant_species == i
    if mask.sum() > 0:
        heatmap[i, :] = ec_filtered[mask].mean(axis=0)

# Normalize rows to sum to 1
heatmap_norm = heatmap / heatmap.sum(axis=1, keepdims=True)

species_names = [
    "Diatom_Ptricornutum", "Diatom_Csimplex", "Chlamydomonas_Cpriscuii",
    "Chlamydomonas_Creindhardtii", "Dinoflagellate_Symbiodiniumsp",
    "Dinoflagellate_Smicroadriaticum", "Dinoflagellate_Dtrenchii",
    "Dinoflagellate_Cgoreaui", "Cyanobacteria_Synechosystis"
]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    heatmap_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=species_names,
    yticklabels=species_names,
    ax=ax,
    vmin=0, vmax=1
)
ax.set_xlabel("Predicted species (mean estimated concentration)")
ax.set_ylabel("Dominant true species")
ax.set_title(f"MLP: Mean predicted concentration\ngrouped by dominant true species (threshold={threshold})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
# contuining from last comment, this works waaay better!

In [ ]:
n_species = len(species_names)
ground_truth = weights
# --- Per-species metrics ---
r2_per_species  = [r2_score(ground_truth[:, i], estimated_concentrations[:, i]) for i in range(n_species)]

# --- Scatter plots: true vs predicted per species ---
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()

for i, (name, ax) in enumerate(zip(species_names, axes)):
    ax.scatter(ground_truth[:, i], estimated_concentrations[:, i], alpha=0.1, s=5, color="steelblue")
    # Perfect prediction line
    lim = [0, max(ground_truth[:, i].max(), estimated_concentrations[:, i].max())]
    ax.plot(lim, lim, "r--", linewidth=1)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel("True", fontsize=7)
    ax.set_ylabel("Predicted", fontsize=7)
    ax.text(0.05, 0.92, f"MAE={per_mixture_mae[i]:.3f}\nR²={r2_per_species[i]:.3f}",
            transform=ax.transAxes, fontsize=7, verticalalignment='top')

plt.suptitle("MLP: True vs Predicted concentration per species", fontsize=12)
plt.tight_layout()
plt.show()
# shows predicted concentration values on y-axis versus the true concentration on x-axis. If it is perfect, it should follow the red line (linear). 
# R^2 determines how well the linear fit is. For noise_level = 0, it is perfect!